In [ ]:
# Pregunta 18 Diseña un agente para un juego de ordenar bloques por colores.

import numpy as np
from collections import defaultdict

class Board:
    def __init__(self):
        self.state = ["amarillo", "verde", "rojo"]
        self.goal = ["rojo", "amarillo", "verde"]

    def valid_moves(self):
        return [0, 1,] 

    def update(self, move):
        if move == 0:
            self.state[0], self.state[1] = self.state[1], self.state[0]
        elif move == 1:
            self.state[1], self.state[2] = self.state[2], self.state[1]
        else:
            raise ValueError("Movimiento ilegal, solo 0 o 1 están permitidos.")

    def is_game_over(self):
        if self.state == self.goal:
            return True
        return False

    def reset(self):
        self.state = ["amarillo", "verde", "rojo"]

In [5]:
tablero_prueba = Board()
print("Tablero inicial:")
print(tablero_prueba.state)
movimientos = tablero_prueba.valid_moves()
print(f"\nMovimientos válidos: {movimientos}")


tablero_prueba.update(1) 
tablero_prueba.update(1)
print(tablero_prueba.state)
print(f"Resultado del juego: {tablero_prueba.is_game_over()}")


Tablero inicial:
['amarillo', 'verde', 'rojo']

Movimientos válidos: [0, 1]
['amarillo', 'verde', 'rojo']
Resultado del juego: False


In [6]:
class Agente:
    def __init__(self, epsilon=0.1, alpha=0.5, gamma=0.6):
        self.q = defaultdict(lambda: np.zeros(2))
        self.eps = epsilon
        self.alpha = alpha
        self.gamma = gamma

    def stado_list(self, state_list):
        return tuple(state_list)

    def choose_action(self, state):
        stado_list = self.stado_list(state)
        if np.random.random() < self.eps:
            return np.random.randint(2)  # exploración
        else:
            return np.argmax(self.q[stado_list])  # explotación

    def learn(self, state, action, reward, next_state, done):
        stado_list = self.stado_list(state)
        next_key = self.stado_list(next_state)
        q_next = 0 if done else max(self.q[next_key])
        td_target = reward + self.gamma * q_next
        self.q[stado_list][action] += self.alpha * (td_target - self.q[stado_list][action])

        

In [ ]:
Agent=Agente()
Agent.stado_list(tablero_prueba.state)

In [7]:
def train(episodes=1000, max_steps=10):
    board = Board()
    agent = Agente(epsilon=0.2, alpha=0.5, gamma=0.9)
    for ep in range(episodes):
        board.reset()
        state = board.state.copy()
        done = False
        steps = 0
        while not done and steps < max_steps:
            action = agent.choose_action(state)
            board.update(action)
            next_state = board.state.copy()
            done = board.is_game_over()
            reward = 10 if done else -0.1
            agent.learn(state, action, reward, next_state, done)
            state = next_state
            steps += 1
    return agent

agente = train(1000)

def test():
    board = Board()
    board.reset()
    state = board.state.copy()
    print("Estado inicial:", state)
    steps = 0
    while not board.is_game_over() and steps < 10:
        # elegir la mejor acción según Q (explotación pura)
        action = np.argmax(agente.q[agente.stado_list(state)])
        board.update(action)
        state = board.state.copy()
        print(f"Paso {steps+1}: intercambio {action} -> {state}")
        steps += 1
    if board.is_game_over():
        print("¡Objetivo alcanzado!")
    else:
        print("No se alcanzó el objetivo en el límite de pasos.")
test()

Estado inicial: ['amarillo', 'verde', 'rojo']
Paso 1: intercambio 1 -> ['amarillo', 'rojo', 'verde']
Paso 2: intercambio 0 -> ['rojo', 'amarillo', 'verde']
¡Objetivo alcanzado!
